# Introducción a Hugging Face: Procesamiento del Lenguaje Natural en Español

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento de Lenguaje Natural
* **Nombre de Trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Objetivo
Introducir de forma práctica el ecosistema de Hugging Face en español, cubriendo desde pipelines básicos de inferencia y conjuntos de datos hasta extracción de texto en PDFs y Question Answering.

## Resultados de aprendizaje
Al final de este notebook vas a poder:
1. Utilizar pipelines pre-entrenados adaptados al español para análisis de sentimientos y emociones.
2. Cargar y procesar conjuntos de datos oficiales desde Hugging Face Hub.
3. Construir una solución integrada que extraiga texto de PDFs y responda preguntas contextuales sobre ellos.



## Terminología clave (Microglosario)

* **✦ Hugging Face Hub:** Repositorio en la nube que actúa como la mayor comunidad global para alojar y compartir modelos, datasets y demos interactivas de IA.
* **✦ AutoModel y AutoTokenizer:** Clases inteligentes que identifican y descargan automáticamente la arquitectura de red y el tokenizador correctos para un modelo dado, facilitando su carga con una única línea de código.
* **✦ Question Answering (QA):** Tarea de NLP que consiste en extraer respuestas precisas a preguntas formuladas a partir de un texto contextual dado.



## 1. Generación de Texto con Salamandra

### Conceptos clave

Los modelos de generación de texto predicen y producen texto nuevo a partir de una semilla inicial (prompt). En español, **Salamandra** es la familia de modelos más moderna y eficiente, desarrollada por el Barcelona Supercomputing Center (BSC-LT).

En este ejercicio utilizaremos **Salamandra-2B-Instruct**, un modelo con 2.25 mil millones de parámetros optimizado para seguir instrucciones. Esta versión es ideal para Google Colab, ya que requiere aproximadamente 4.5GB de memoria GPU.

**Nota para Google Colab**: Si experimentás problemas de memoria, asegurate de activar la GPU en Runtime → Change runtime type → Hardware accelerator → GPU.

### Ejercicio: Generar texto en español

Crea una canalización de generación de texto y úsala para completar una frase inicial. Experimenta con los parámetros `max_new_tokens` (cantidad de tokens a generar) y observa cómo el modelo continúa el texto.

In [ ]:
from transformers import pipeline

In [ ]:
# Crear la canalización de generación de texto con Salamandra-2B-Instruct
# Modelo verificado: BSC-LT/salamandra-2b-instruct
# Estado: Activo (3,055 descargas/mes), 2.25B parámetros (~4.5GB VRAM)
# Compatible con Google Colab Free Tier
generador = pipeline(
    task="text-generation",
    model="BSC-LT/salamandra-2b-instruct"
)

In [ ]:
# Generar texto a partir de una frase inicial en español
# Parámetros utilizados:
#   - max_new_tokens: cantidad máxima de tokens nuevos a generar
#   - num_return_sequences: cantidad de variaciones diferentes (opcional)
resultado = generador(
    "La inteligencia artificial en la educación",
    max_new_tokens=50,
    num_return_sequences=2
)

# Mostrar los resultados
print("Textos generados:\n")
for i, resultado in enumerate(resultado, 1):
    print(f"Secuencia {i}:")
    print(resultado['generated_text'])
    print()

## 2. Carga de Conjuntos de Datos

### Conceptos clave

Los conjuntos de datos (*datasets*) son colecciones de ejemplos estructurados que utilizamos para entrenar, validar y evaluar modelos de aprendizaje automático. Hugging Face ofrece una biblioteca especializada llamada `datasets` que facilita acceder a miles de conjuntos de datos públicos y gestionar nuestros datos de forma eficiente.

En este ejercicio, cargarás la división de validación del conjunto MMLU-Pro, un conjunto de evaluación de referencia que contiene preguntas de opción múltiple en múltiples disciplinas.

### Ejercicio: Cargar un conjunto de datos

Utiliza la función `load_dataset` para cargar la división "validation" del conjunto de datos "TIGER-Lab/MMLU-Pro". Luego, examina la estructura y el contenido del conjunto cargado.

In [ ]:
from datasets import load_dataset

# Cargar el conjunto de datos de Wikipedia en español
# Utilizamos una muestra pequeña para mayor velocidad (primeros 1000 ejemplos)
wikipedia = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:1000]")

# Mostrar información del conjunto cargado
print("Conjunto de datos de Wikipedia cargado:")
print(wikipedia)

## 3. Manipulación y Filtrado de Conjuntos de Datos

### Conceptos clave

Antes de utilizar un conjunto de datos, frecuentemente necesitamos transformarlo para adaptarlo a nuestras necesidades específicas. Las operaciones más comunes son el filtrado (seleccionar filas según criterios) y la selección (extraer subconjuntos de datos).

Hugging Face utiliza tipos de archivo Arrow para almacenar datos de forma eficiente, lo que significa que las manipulaciones se realizan de manera diferente a las listas tradicionales de Python. La buena noticia es que la biblioteca proporciona métodos convenientes para estas operaciones.

### Ejercicio: Filtrar y seleccionar datos

Filtra un conjunto de datos para obtener solo las filas que contengan el término "fútbol" en la columna "text". Luego, selecciona un único ejemplo del conjunto filtrado.

Nota: Para este ejercicio, utilizaremos un conjunto de datos de Wikipedia que ya debe estar cargado como la variable `wikipedia`.

In [ ]:
# Filtrar el conjunto de datos para obtener documentos que contengan "fútbol"
# Utilizamos una función lambda para definir el criterio de filtrado
filtrado = wikipedia.filter(lambda fila: "fútbol" in fila["text"])

# Seleccionar un único ejemplo del conjunto filtrado
# select(range(1)) extrae los primeros 1 elemento(s)
ejemplo = filtrado.select(range(1))

# Mostrar el texto del ejemplo seleccionado
print("Ejemplo de texto sobre fútbol:")
print("=" * 50)
print(ejemplo[0]["text"])
print("=" * 50)

## 4. Clasificación de Textos: Análisis de Sentimientos

### Conceptos clave

La clasificación de textos es el proceso de asignar etiquetas o categorías a un texto. Esta tarea tiene múltiples aplicaciones: análisis de sentimientos, detección de spam, identificación de idiomas, categorización de noticias, entre otras.

El análisis de sentimientos es una de las aplicaciones más populares de clasificación. Identifica la actitud emocional expresada en un texto, lo cual es fundamental para entender opiniones de clientes, reacciones en redes sociales y feedback en general.

**RoBERTuito** es un modelo RoBERTa entrenado específicamente con 500 millones de tweets en español. Esto lo convierte en una opción excelente para analizar texto informal, coloquial y conversacional. El modelo clasifica textos en tres categorías: positivo (POS), negativo (NEG) y neutral (NEU).

**Ventajas para Colab**: Con solo 110 millones de parámetros (~400MB-1GB VRAM), RoBERTuito es extremadamente liviano y rápido en Google Colab.

### Ejercicio: Analizar el sentimiento de reseñas

Crea una canalización de clasificación de textos para análisis de sentimientos. Úsala para evaluar la actitud emocional en diferentes textos en español y observa cómo el modelo asigna etiquetas con valores de confianza.

In [ ]:
from transformers import pipeline

# Crear la canalización de análisis de sentimientos con RoBERTuito
# Modelo verificado: pysentimiento/robertuito-sentiment-analysis
# Estado: Activo (1.7M descargas/mes), 110M parámetros (~400MB-1GB VRAM)
# Macro F1: 0.705, entrenado en 500M tweets españoles
# Excelente para Google Colab
analizador = pipeline(
    task="text-classification",
    model="pysentimiento/robertuito-sentiment-analysis"
)

In [ ]:
# Analizar el sentimiento de una reseña
resultado = analizador("Excelente curso, muy recomendado para aprender PLN")

# Mostrar los resultados
print("Análisis de Sentimientos:")
print(f"Texto: 'Excelente curso, muy recomendado para aprender PLN'")
print(f"Resultado: {resultado}")
print(f"\nInterpretación:")
print(f"Sentimiento: {resultado[0]['label']}")
print(f"Confianza: {resultado[0]['score']:.4f}")

## 5. Análisis de Emociones: Clasificación Multiclase

### Conceptos clave

Mientras que el análisis de sentimientos clasifica textos en positivo/negativo/neutral, el análisis de emociones va más allá e identifica emociones específicas como alegría, tristeza, enojo, miedo, sorpresa, entre otras.

Cada modelo de clasificación está entrenado con objetivos específicos y aprende a reconocer patrones distintos. Por eso cambiamos de modelo según la tarea que necesitamos resolver.

**RoBERTuito para emociones** es una variante del modelo base entrenada específicamente para detectar 7 emociones diferentes en textos en español. Esto es especialmente útil para análisis más granular de reacciones y opiniones.

**Compatible con Colab**: ~110M parámetros, igual de liviano que la versión de sentimientos.

### Ejercicio: Detectar emociones en textos

Crea una canalización de clasificación de emociones y úsala para identificar la emoción predominante en diferentes textos. Compara cómo diferentes emociones tienen scores distintos.

In [ ]:
# Crear la canalización de análisis de emociones
# Modelo verificado: pysentimiento/robertuito-emotion-analysis
# Detecta 7 emociones: others, joy, sadness, anger, surprise, disgust, fear
clasificador_emociones = pipeline(
    task="text-classification",
    model="pysentimiento/robertuito-emotion-analysis"
)

In [ ]:
# Texto para analizar
texto = "¡No puedo creer que ganamos el campeonato! ¡Increíble!"

# Predecir la emoción
resultado = clasificador_emociones(texto)

# Mostrar el resultado
print("Análisis de Emociones:")
print(f"Texto: '{texto}'")
print(f"\nResultado: {resultado}")
print(f"\nInterpretación:")
print(f"Emoción detectada: {resultado[0]['label']}")
print(f"Confianza: {resultado[0]['score']:.4f}")

## 6. Clasificación con Cero Ejemplos (Zero-Shot Classification)

### Conceptos clave

La clasificación con cero ejemplos (*zero-shot classification*) permite a un modelo asignar textos a categorías sin haber sido entrenado específicamente con esas categorías. Es una capacidad poderosa que emergió en modelos modernos.

En lugar de limitarnos a las etiquetas en las que fue entrenado el modelo, podemos definir nuestras propias categorías y dejar que el modelo determine cuál es la más probable. Esto es especialmente útil cuando tenemos categorías personalizadas o dinámicas.

**XLM-RoBERTa-Large** es un modelo multilingüe entrenado en 100 idiomas, incluyendo español. Aunque es más pesado que otros modelos en este cuaderno (560 millones de parámetros), funciona perfectamente en Google Colab y ofrece resultados excelentes para clasificación zero-shot.

**Memoria en Colab**: ~2.4GB VRAM para inferencia, perfectamente compatible con el tier gratuito.

### Ejercicio: Clasificar noticias en categorías personalizadas

Crea un clasificador zero-shot usando el modelo XLM-RoBERTa. Define tus propias categorías ("política", "ciencia", "deportes") y clasifica un titular de noticias en una de estas categorías.

In [ ]:
# Texto del titular de noticia a clasificar
texto = "Robots impulsados por IA asisten en cirugías cerebrales complejas con precisión."

In [ ]:
# Crear la canalización de clasificación zero-shot
# Modelo verificado: FacebookAI/xlm-roberta-large
# Estado: Activo (2.9M descargas/mes), 560M parámetros
# Compatible con Google Colab
clasificador = pipeline(
    task="zero-shot-classification",
    model="FacebookAI/xlm-roberta-large"
)

In [ ]:
# Definir nuestras propias categorías
categorias = ["política", "ciencia", "deportes", "sociedad", "tecnologia"]

In [ ]:
# Realizar la predicción
resultado = clasificador(texto, categorias)

# Mostrar el resultado de forma clara
print("Clasificación Zero-Shot:")
print(f"Texto: '{texto}'")
print(f"Categorías disponibles: {categorias}\n")

print("Resultados (ordenados por probabilidad):")
for i, (etiqueta, puntuacion) in enumerate(zip(resultado['labels'], resultado['scores']), 1):
    barra = "█" * int(puntuacion * 20)
    print(f"{i}. {etiqueta.upper()}: {puntuacion:.4f} {barra}")

print(f"\nCategoría más probable: {resultado['labels'][0].upper()}")

## 7. Resumen de Textos (Summarization)

### Conceptos clave

El resumen de textos es el proceso de condensar contenidos extensos en versiones más breves que conserven la información más importante. Existen dos enfoques principales:

- **Resumen extractivo**: selecciona frases clave del texto original sin modificarlas.
- **Resumen abstractivo**: genera nuevas frases que resumen las ideas principales, como si una persona lo escribiera de su puño y letra.

En este ejercicio utilizaremos resumen abstractivo con **mT5-base-DACSA-ES**, un modelo mT5 (multilingual T5) ajustado específicamente para resumir noticias en español. Fue entrenado con 1.8 millones de pares artículo-resumen del dataset DACSA.

**Compatible con Colab**: Basado en mT5-base (~580M parámetros), requiere ~2-3GB VRAM, funciona bien en Colab.

### Ejercicio: Resumir un texto largo

Crea una canalización de resumen abstractivo y úsala para condensar un texto sobre un tema. Luego, compara la longitud del texto original con el del resumen.

In [ ]:
# Crear la canalización de resumen abstractivo
# Modelo verificado: ELiRF/mt5-base-dacsa-es
# Basado en mT5, entrenado con 1.8M pares artículo-resumen en español
# ~580M parámetros (~2-3GB VRAM), compatible con Colab
resumidor = pipeline(
    task="summarization",
    model="ELiRF/mt5-base-dacsa-es"
)

In [ ]:
# Texto original a resumir
texto_original = """
La inteligencia artificial está transformando la educación de múltiples formas.
Los sistemas de tutoría inteligente pueden personalizar el aprendizaje según las
necesidades individuales de cada estudiante, adaptando el contenido y el ritmo
a su nivel de comprensión. Además, las herramientas de evaluación automática
permiten a los docentes ahorrar tiempo en correcciones y dedicar más esfuerzo
a la planificación pedagógica. Los chatbots educativos ofrecen soporte disponible
las 24 horas del día a los estudiantes que tienen preguntas fuera del horario
escolar. Por último, los sistemas de análisis de datos ayudan a identificar
estudiantes en riesgo de abandono escolar para intervenir a tiempo.
"""

# Resumir el texto
texto_resumido = resumidor(texto_original, max_length=60, min_length=20)

# Comparar las longitudes
longitud_original = len(texto_original)
longitud_resumen = len(texto_resumido[0]['summary_text'])

# Mostrar comparación
print("Resumen de Texto:")
print("=" * 60)
print(f"\nTexto Original ({longitud_original} caracteres):")
print(texto_original.strip())

print(f"\n\nTexto Resumido ({longitud_resumen} caracteres):")
print(texto_resumido[0]['summary_text'])

# Calcular la reducción
reduccion = (1 - longitud_resumen / longitud_original) * 100
print(f"\n\nReducción lograda: {reduccion:.1f}%")

## 8. Ajuste de Parámetros en Resumen

### Conceptos clave

Los modelos de resumen ofrecen parámetros para controlar la longitud del output. Los dos más importantes son:

- **`min_length`**: número mínimo de tokens que el modelo debe generar. Útil para garantizar un resumen con extensión mínima.
- **`max_length`**: número máximo de tokens que el modelo puede generar. Útil para limitar la longitud.

Estos parámetros son valiosos cuando tenemos restricciones de espacio (poco almacenamiento), buscamos mejorar la legibilidad, o queremos garantizar una cierta calidad y detalle en el resumen.

### Ejercicio: Generar resúmenes de diferentes longitudes

Crea dos canalizaciones de resumen: una para generar resúmenes breves (máximo 30 tokens) y otra para generar resúmenes extensos (50-100 tokens). Aplica ambas al mismo texto y compara los resultados.

In [ ]:
# Crear un resumidor para textos BREVES (máximo 30 tokens)
resumidor_breve = pipeline(
    task="summarization",
    model="ELiRF/mt5-base-dacsa-es"
)

In [ ]:
# Resumir con extensión breve
resumen_breve = resumidor_breve(texto_original, max_length=30, min_length=10)

print("RESUMEN BREVE (10-30 tokens):")
print("=" * 60)
print(resumen_breve[0]["summary_text"])
print()

# Crear un resumidor para textos LARGOS (50-100 tokens)
resumidor_largo = pipeline(
    task="summarization",
    model="ELiRF/mt5-base-dacsa-es"
)

# Resumir con extensión larga
resumen_largo = resumidor_largo(texto_original, max_length=100, min_length=50)

print("\nRESUMEN LARGO (50-100 tokens):")
print("=" * 60)
print(resumen_largo[0]["summary_text"])
print()

# Comparación de longitudes
print("COMPARACIÓN:")
print("=" * 60)
print(f"Resumen breve: {len(resumen_breve[0]['summary_text'])} caracteres")
print(f"Resumen largo: {len(resumen_largo[0]['summary_text'])} caracteres")

## 9. Tokenización de Texto con AutoTokenizer

### Conceptos clave

La tokenización es el proceso de dividir un texto en unidades más pequeñas llamadas tokens. Un token puede ser una palabra, una subpalabra, o incluso un carácter, dependiendo del tokenizador.

Los tokenizadores son componentes fundamentales en procesamiento del lenguaje natural. No solo dividen el texto, sino que también:

- Limpian y normalizan el texto.
- Convierten palabras en identificadores numéricos.
- Manejan caracteres especiales y acentos.
- Adaptan el texto al formato que espera el modelo.

`AutoTokenizer` es una clase que automáticamente carga el tokenizador correcto según el modelo que especifiques. Esto garantiza coherencia entre el tokenizador y el modelo.

**BETO** (BERT en Español) utiliza WordPiece tokenization, dividiendo palabras en subunidades para manejar vocabulario abierto eficientemente.

### Ejercicio: Tokenizar texto con AutoTokenizer

Carga un tokenizador automáticamente desde un modelo preentrenado. Utiliza el método `tokenize()` para dividir una oración en tokens y observa cómo el tokenizador procesa el texto en español.

In [ ]:
from transformers import AutoTokenizer

In [ ]:
# Cargar el tokenizador para el modelo BETO
# AutoTokenizer detecta automáticamente qué tipo de tokenizador usar
# Modelo: dccuchile/bert-base-spanish-wwm-cased
tokenizador = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

# Tokenizar el texto usando el método tokenize()
texto = "La inteligencia artificial está revolucionando la educación"
tokens = tokenizador.tokenize(texto)

# Mostrar los resultados
print("Tokenización de Texto:")
print("=" * 60)
print(f"Texto original:")
print(f"  {texto}\n")

print(f"Tokens generados ({len(tokens)} tokens):")
print(f"  {tokens}\n")

# Mostrar tokens con índices
print("Tokens con sus identificadores numéricos:")
ids_tokens = tokenizador.convert_tokens_to_ids(tokens)
for token, id_token in zip(tokens, ids_tokens):
    print(f"  '{token}' → ID {id_token}")

## 10. Uso de AutoClasses: Integrando Modelo y Tokenizador

### Conceptos clave

Hasta ahora utilizamos canalizaciones (*pipelines*) que nos ofrecen una interfaz simple pero abstraen mucho de lo que sucede internamente. A veces necesitamos mayor control sobre el proceso.

Las `AutoClasses` permiten cargar componentes individuales:

- **`AutoTokenizer`**: carga el tokenizador correcto.
- **`AutoModel`**: carga el modelo base según la tarea.
- **`AutoModelForSequenceClassification`**: carga un modelo específico para clasificación de secuencias.

Combinando tokenizador y modelo manualmente, ganamos flexibilidad. Luego, podemos envolver estos componentes en una canalización para obtener lo mejor de ambos mundos: control y comodidad.

### Ejercicio: Crear una canalización personalizada con AutoClasses

Descarga un modelo de análisis de sentimientos y su tokenizador usando `AutoModelForSequenceClassification` y `AutoTokenizer`. Crea una canalización manual y úsala para análisis de sentimientos en español.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline
)

# Descargar el modelo y el tokenizador del repositorio de Hugging Face
# Usamos RoBERTuito para análisis de sentimientos en español
mi_modelo = AutoModelForSequenceClassification.from_pretrained(
    "pysentimiento/robertuito-sentiment-analysis"
)
mi_tokenizador = AutoTokenizer.from_pretrained(
    "pysentimiento/robertuito-sentiment-analysis"
)

# Crear la canalización usando el modelo y tokenizador que descargamos
mi_canalización = pipeline(
    task="sentiment-analysis",
    model=mi_modelo,
    tokenizer=mi_tokenizador
)

# Analizar el sentimiento de una oración
texto_evaluado = "Este curso de PLN en español es excelente, aprendí muchísimo."
resultado = mi_canalización(texto_evaluado)

# Mostrar los resultados
print("Análisis de Sentimientos con AutoClasses:")
print("=" * 60)
print(f"Texto analizado: '{texto_evaluado}'")
print(f"\nResultado completo: {resultado}\n")

print("Interpretación:")
print(f"  Sentimiento: {resultado[0]['label']}")
print(f"  Puntuación de confianza: {resultado[0]['score']:.4f}")

## 11. Extracción de Texto desde PDF con PyPDF

### Conceptos clave

Los archivos PDF (*Portable Document Format*) son un formato estándar para compartir documentos. Cuando queremos procesar información contenida en PDFs con herramientas de PLN, primero necesitamos extraer el texto.

La biblioteca `PyPDF` (anteriormente PyPDF2) permite leer y manipular archivos PDF de forma sencilla. Podemos iterar sobre cada página del documento y extraer su contenido textual.

Este paso es fundamental cuando trabajamos con documentos reales como políticas de empresa, artículos académicos, reportes o cualquier documento en formato PDF.

### Ejercicio: Extraer texto de un PDF

Lee un archivo PDF página por página y combina todo el contenido en una única cadena de texto. Este texto extraído será el insumo para tareas posteriores de procesamiento como preguntas y respuestas.

Nota: Para este ejercicio, asumimos que tienes un archivo PDF disponible (por ejemplo, "US_Employee_Policy.pdf"). Si no tenés uno, podés usar cualquier PDF que tengas a mano.

In [ ]:
%pip install pypdf

In [ ]:
# Solo necesario si estás en Google Colab y querés usar Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pypdf import PdfReader
import os

# Ruta del archivo PDF (intentamos primero local, si no Colab/Drive)
ruta_pdf = "datos/ARG_Employee_Policy.pdf"
if not os.path.exists(ruta_pdf):
    ruta_pdf = "/content/drive/MyDrive/Clases/000 - 2DO Cuatrimestre/HABLA/010 - Hugging Face/ARG_Employee_Policy.pdf"

# Crear un lector de PDF para acceder al archivo
lector = PdfReader(ruta_pdf)


The `pypdf` library has been installed. Now you can run the cell above to extract text from the PDF.

## 12. Preguntas y Respuestas sobre Documentos Extraídos

### Conceptos clave

La tarea de preguntas y respuestas (*question answering* o QA) permite buscar y extraer respuestas específicas de un texto. Esto es especialmente útil cuando tenemos documentos largos (como el que extraímos del PDF) y queremos localizar información precisa sin leerlo completamente.

Un modelo QA:
1. Recibe una pregunta y un contexto (texto).
2. Identifica dónde en el contexto se encuentra la respuesta.
3. Extrae y devuelve ese fragmento específico.

**RoBERTa-Base-BNE-SQAC** es un modelo RoBERTa entrenado por BSC-LT específicamente para preguntas y respuestas en español, usando el dataset SQAC (Spanish Question Answering Corpus).

### Ejercicio: Buscar respuestas en un documento

Crea una canalización de preguntas y respuestas usando un modelo especializado en español. Úsala para obtener respuestas específicas del documento PDF que extrajimos anteriormente.

Nota: Para este ejercicio, utilizaremos la variable `texto_documento` que contiene el texto extraído del PDF.

In [ ]:
# Cargar la canalización de preguntas y respuestas en español
# Modelo: BSC-LT/roberta-base-bne-sqac
# Entrenado en SQAC (Spanish Question Answering Corpus)

canalización_qa = pipeline("question-answering", model="timpal0l/mdeberta-v3-base-squad2")


In [ ]:
# Definir una pregunta (adapta según tu documento)
pregunta = "¿Cuál es el período de aviso para la renuncia?"

# Obtener la respuesta usando el modelo QA
resultado = canalización_qa(question=pregunta, context=texto_documento)

# Mostrar los resultados
print("Búsqueda de Respuestas en Documentos:")
print("=" * 60)
print(f"Pregunta: '{pregunta}'\n")

print(f"Respuesta encontrada:")
print(f"  {resultado['answer']}\n")

print(f"Información adicional:")
print(f"  Confianza: {resultado['score']:.4f}")
print(f"  Posición en el texto: inicio={resultado['start']}, fin={resultado['end']}\n")

# Mostrar el contexto alrededor de la respuesta
inicio = max(0, resultado['start'] - 100)
fin = min(len(texto_documento), resultado['end'] + 100)
print(f"Contexto:")
print(f"  ...{texto_documento[inicio:fin]}...")

---

## Glosario de Términos

Este glosario recoge los conceptos clave introducidos en el cuaderno, con definiciones claras orientadas al contexto de modelos en español.

**AutoTokenizer**: Clase de Hugging Face que carga automáticamente el tokenizador correcto para un modelo preentrenado, garantizando compatibilidad entre el procesamiento de texto y el modelo.

**BETO**: BERT en Español. Modelo BERT entrenado específicamente en corpus españoles, ideal para tareas de clasificación y extracción de información en nuestro idioma.

**BSC-LT**: Barcelona Supercomputing Center - Language Technologies. Institución líder en desarrollo de modelos de PLN para español y lenguas cooficiales de España.

**Canalización (Pipeline)**: Interfaz simplificada de Hugging Face que encadena automáticamente los pasos de tokenización, predicción y postprocesamiento en una sola función fácil de usar.

**Clasificación de textos**: Tarea de PLN que asigna etiquetas o categorías predefinidas a un texto. Ejemplos: análisis de sentimientos (positivo/negativo), detección de spam, categorización de noticias.

**Generación de texto**: Tarea de PLN que produce texto nuevo a partir de una semilla inicial (prompt), prediciendo tokens sucesivamente basándose en patrones aprendidos durante el entrenamiento.

**NASES**: News Abstractive Summarization for Spanish. Modelo especializado en resumen abstractivo de noticias en español. Nota: tiene un bug conocido en la API de inferencia.

**Question Answering (QA)**: Tarea que extrae respuestas específicas de un texto dado. El modelo recibe una pregunta y un contexto, e identifica dónde en el contexto se encuentra la información solicitada.

**Resumen abstractivo**: Generación de un resumen que parafrasea y reformula las ideas principales del texto original, en lugar de simplemente extraer frases existentes.

**RoBERTa**: Robustly Optimized BERT Pretraining Approach. Variante mejorada de BERT con mejor rendimiento en múltiples tareas.

**RoBERTuito**: Modelo RoBERTa entrenado específicamente con 500 millones de tweets en español, optimizado para lenguaje informal y conversacional.

**Salamandra**: Familia de modelos de lenguaje desarrollados por BSC-LT, entrenados desde cero en español y otras lenguas. Disponibles en versiones de 2B, 7B y 40B parámetros.

**SQAC**: Spanish Question Answering Corpus. Conjunto de datos de preguntas y respuestas en español utilizado para entrenar modelos QA.

**Token**: Unidad mínima de procesamiento en PLN. Puede ser una palabra completa, una subpalabra (ej: "aprendizaje" → "aprendi", "##zaje") o un carácter, según el tokenizador.

**Tokenización**: Proceso de dividir un texto en tokens y convertirlos al formato numérico que los modelos pueden procesar. Incluye normalización, limpieza y codificación.

**Transformers**: Biblioteca de Hugging Face que proporciona acceso a miles de modelos preentrenados, utilidades de procesamiento y herramientas para facilitar el trabajo con PLN.

**XLM-RoBERTa**: Cross-lingual Language Model RoBERTa. Modelo multilingüe entrenado en 100 idiomas, excelente para tareas que requieren comprensión de múltiples lenguas.

**Zero-shot classification**: Capacidad de clasificar textos en categorías nuevas sin haber sido entrenado específicamente con ellas. El modelo generaliza su comprensión para aplicarla a etiquetas arbitrarias.

## Preguntas Frecuentes

En esta sección respondemos a las preguntas más comunes que surgen al trabajar con Hugging Face y modelos de PLN en español.

### ¿Por qué usar modelos específicos para español en lugar de modelos en inglés?

Los modelos entrenados en español comprenden mejor las características específicas de nuestro idioma: conjugaciones verbales complejas, uso de acentos, estructura gramatical, modismos y expresiones coloquiales. Esto se traduce en resultados significativamente mejores para tareas en español.

### ¿Estos modelos funcionan en Google Colab gratis?

Sí, todos los modelos en este cuaderno fueron seleccionados específicamente para ser compatibles con la GPU gratuita de Google Colab (12-16GB VRAM). Salamandra-2B, RoBERTuito, BETO y XLM-RoBERTa funcionan sin problemas. Solo evita la versión de 7B parámetros de Salamandra en el tier gratuito.

### ¿Los modelos funcionan offline una vez descargados?

Sí, completamente. La primera vez que ejecutas el código, Hugging Face descarga automáticamente el modelo y lo guarda en caché local. Después de eso, podés trabajar sin conexión a Internet. Los modelos se guardan en `~/.cache/huggingface/`.

### ¿Qué significa el "score" o "confianza" en los resultados?

El score representa la probabilidad o confianza que el modelo asigna a su predicción, en una escala de 0 a 1. Un score de 0.9 significa 90% de confianza. Scores bajos (< 0.3-0.4) generalmente indican que el modelo está muy inseguro y la predicción puede ser incorrecta.

### ¿Puedo usar estos modelos para proyectos comerciales?

Depende de la licencia de cada modelo. Salamandra usa licencia Apache 2.0 (permisiva), pero siempre verifica la licencia en la página del modelo en Hugging Face antes de usar en producción.

### ¿Cómo puedo entrenar o ajustar estos modelos con mis propios datos?

Hugging Face ofrece utilidades para *fine-tuning* (ajuste fino). Podés tomar un modelo preentrenado y entrenarlo con tus datos específicos usando la clase `Trainer`. Esto requiere datos etiquetados y conocimientos de entrenamiento de modelos.

### ¿Qué hacer si obtengo resultados pobres o incorrectos?

Considera estas opciones:
- **Cambia a un modelo más grande o especializado**: Por ejemplo, si Salamandra-2B no funciona bien, prueba con la versión de 7B (requiere más memoria)
- **Pre-procesa tu texto**: Limpia caracteres extraños, normaliza espacios, corrige errores evidentes
- **Ajusta los parámetros**: Experimenta con `max_new_tokens`, `temperature`, etc.
- **Proporciona más contexto**: Para QA y clasificación, más contexto relevante mejora los resultados
- **Reformula la entrada**: A veces reformular la pregunta o prompt produce mejores resultados

### ¿Qué modelo debo elegir para mi tarea específica?

- **Generación de texto**: Salamandra-2B-Instruct
- **Análisis de sentimientos** (especialmente redes sociales): RoBERTuito
- **Clasificación en categorías personalizadas**: XLM-RoBERTa con zero-shot
- **Tokenización y clasificación general**: BETO
- **Preguntas y respuestas**: RoBERTa-Base-BNE-SQAC

### ¿Puedo combinar múltiples modelos en un mismo proyecto?

¡Por supuesto! Es una práctica común usar diferentes modelos especializados para diferentes partes de tu aplicación. Por ejemplo: RoBERTuito para sentimientos, Salamandra para generación, y RoBERTa-SQAC para preguntas.

### ¿Qué pasa si mi pregunta no tiene respuesta en el contexto (para QA)?

El modelo intentará encontrar algo de todos modos, pero devolverá un score muy bajo. Siempre verifica el score: si es inferior a 0.3, probablemente la información no está en el contexto proporcionado.

### ¿Cómo activo la GPU en Google Colab?

Ve a **Runtime → Change runtime type → Hardware accelerator → GPU**. Luego reinicia el runtime. Podés verificar que la GPU esté activa ejecutando `!nvidia-smi` en una celda.

### ¿Hay modelos aún mejores para español que los presentados aquí?

El campo está en constante evolución. Los modelos presentados son excelentes a octubre de 2025 y están activamente mantenidos. Para tareas específicas, siempre revisa la colección de modelos en español en Hugging Face: https://huggingface.co/models?language=es